# Study 891 — Insurance Float Engine 🛡️

**P&C insurers earn on "float" — premiums held before claims are paid. It made Buffett.
Does a plain basket of insurers turn it into a market-beating edge?**

A property-and-casualty insurer collects your premium today and pays the claim later; the money
in between — the **float** — it invests for itself. Near-zero-cost leverage: the compounding
engine behind Berkshire. The folklore extrapolates: *so a broad insurer basket must be a quiet,
structurally-advantaged compounder.* We race two liquid wrappers — **KIE** (SPDR S&P Insurance,
equal-weight) and **IAK** (iShares U.S. Insurance) — against **SPY**, both **excess-of-cash**
(minus **BIL**), over 229 months (2007-06-30 → 2026-06-30), with **KBE** (banks) as the
control that asks: *float premium, or just financial-sector beta?*

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `3a54dbc09ab6`); the live
cell runs the fast synthetic control. Short sample: 19 years, one of them the GFC — named on the
Signal axis.*


## 1. The race, in one table

Both legs are measured *excess-of-cash* (minus BIL) before we annualise the Sharpe, so a high short rate can't flatter anyone. If the float were a structural edge, the insurer baskets should out-Sharpe the market. They don't — they trail it, at higher volatility and a deeper drawdown.

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n': 229, 'fp': '3a54dbc09ab6', 'kie_cagr': 7.86, 'kie_vol': 21.73, 'kie_sharpe': 0.398, 'kie_dd': -69.7, 'iak_cagr': 6.8, 'iak_vol': 20.85, 'iak_sharpe': 0.359, 'iak_dd': -72.2, 'spy_cagr': 10.68, 'spy_vol': 15.58, 'spy_sharpe': 0.644, 'spy_dd': -50.8, 'kbe_cagr': 3.21, 'kbe_vol': 27.64, 'kbe_sharpe': 0.209, 'kbe_dd': -76.6, 'kie_adv': -0.246, 'kie_diff': -1.39, 'kie_tdiff': -0.49, 'iak_adv': -0.285, 'iak_diff': -2.56, 'iak_tdiff': -0.9, 'kie_ci': (-0.06, 0.966), 'iak_ci': (-0.116, 0.949), 'spy_ci': (0.18, 1.182), 'kie_capm_a': -2.48, 'kie_capm_ta': -0.84, 'kie_capm_b': 1.109, 'iak_capm_a': -3.1, 'iak_capm_ta': -0.95, 'iak_capm_b': 1.053, 'kie_two_a': -0.11, 'kie_two_ta': -0.04, 'kie_two_load': 0.357, 'kie_two_tload': 6.27, 'iak_two_a': -0.96, 'iak_two_ta': -0.34, 'iak_two_load': 0.322, 'iak_two_tload': 5.81, 'kie_kbe': 2.89, 'kie_kbe_t': 0.96, 'iak_kbe': 1.71, 'iak_kbe_t': 0.58, 'era_gfc': 0.118, 'era_1015': -0.125, 'era_1620': -0.374, 'era_2126': -0.157, 'era_post': -0.207, 'rot_net': 9.51, 'rot_sharpe': 0.49, 'rot_mkt': 11.41, 'rot_ins': 10.03, 'rot_switches': 30, 'iso_gross': -1.39, 'iso_tg': -0.49, 'iso_net': -1.89, 'iso_tn': -0.66, 'iso_charge': 0.5, 'syn_null_adv': 0.092, 'syn_null_ta': 1.48, 'syn_edge_adv': 0.305, 'syn_edge_ta': 4.34, 'syn_load': 0.387, 'syn_tload': 10.7}
print('basket   CAGR    vol   exSharpe   maxDD')
for t in ['kie','iak','spy','kbe']:
    print(f"{t.upper():5s} {R[t+'_cagr']:6.2f}% {R[t+'_vol']:6.2f}% "
          f"  {R[t+'_sharpe']:+.3f}   {R[t+'_dd']:6.1f}%")
print()
print(f"KIE vs SPY: Sharpe advantage {R['kie_adv']:+.3f}  "
      f"(KIE-SPY {R['kie_diff']:+.2f}%/yr, HAC t={R['kie_tdiff']:+.2f})")
print(f"IAK vs SPY: Sharpe advantage {R['iak_adv']:+.3f}  "
      f"(IAK-SPY {R['iak_diff']:+.2f}%/yr, HAC t={R['iak_tdiff']:+.2f})")

basket   CAGR    vol   exSharpe   maxDD
KIE     7.86%  21.73%   +0.398    -69.7%
IAK     6.80%  20.85%   +0.359    -72.2%
SPY    10.68%  15.58%   +0.644    -50.8%
KBE     3.21%  27.64%   +0.209    -76.6%

KIE vs SPY: Sharpe advantage -0.246  (KIE-SPY -1.39%/yr, HAC t=-0.49)
IAK vs SPY: Sharpe advantage -0.285  (IAK-SPY -2.56%/yr, HAC t=-0.90)


## 2. Where did the 'edge' go? It was financial-sector beta all along

Regress the insurer's excess return on the market **plus** one financial-sector factor (banks minus market). If float were a real premium it would survive as a positive alpha. Instead the alpha vanishes to zero and the bank factor loads up hard:

In [2]:
print(f"KIE CAPM alpha (market only): {R['kie_capm_a']:+.2f}%/yr (t={R['kie_capm_ta']:+.2f})")
print(f"KIE alpha after adding the bank factor: {R['kie_two_a']:+.2f}%/yr "
      f"(t={R['kie_two_ta']:+.2f})  <- collapses to zero")
print(f"   ...while the bank-sector loading is {R['kie_two_load']:+.3f} "
      f"(t={R['kie_two_tload']:+.2f})  <- big and significant")
print()
print('Translation: the insurer basket IS financial-sector beta.')
print('The float is real economics, but at the traded-basket level it is not a')
print('distinct, market-beating premium.')

KIE CAPM alpha (market only): -2.48%/yr (t=-0.84)
KIE alpha after adding the bank factor: -0.11%/yr (t=-0.04)  <- collapses to zero
   ...while the bank-sector loading is +0.357 (t=+6.27)  <- big and significant

Translation: the insurer basket IS financial-sector beta.
The float is real economics, but at the traded-basket level it is not a
distinct, market-beating premium.


## 3. Is the machinery honest? A live synthetic control

We plant a *real* +4 %/yr float edge in a seeded toy world (on top of the same market + bank beta) and check the detector recovers it — and stays silent on the null (edge = 0, only sector beta). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from insurance_float import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(edge_ann=0.0, seed=891))
edge = st.synthetic_detect(data.synthetic_world(edge_ann=0.04, seed=891))
print(f"null  (no float edge): Sharpe adv {null['advantage']:+.3f}, "
      f"CAPM alpha t={null['capm_t_alpha']:+.2f}  (should stay quiet)")
print(f"planted +4%/yr edge  : Sharpe adv {edge['advantage']:+.3f}, "
      f"CAPM alpha t={edge['capm_t_alpha']:+.2f}  (should light up)")
print(f"bank loading detected in both: {edge['load_bank']:+.3f} "
      f"(t={edge['t_load_bank']:+.2f}) — the confound is always found")

null  (no float edge): Sharpe adv +0.092, CAPM alpha t=+1.48  (should stay quiet)
planted +4%/yr edge  : Sharpe adv +0.305, CAPM alpha t=+4.34  (should light up)
bank loading detected in both: +0.387 (t=+10.70) — the confound is always found


## 4. The honest verdict

**Signal — None.** The claimed edge over the market is absent, and the sign runs the wrong way: KIE/IAK excess Sharpe **0.40 / 0.36** trails SPY's **0.64**; the advantage is **-0.25 / -0.28** with the return difference statistically zero, CAPM alpha is **negative**, and one financial-sector factor drives the alpha to **-0.11 %/yr (t = -0.04)**. **Tradability — Mirage:** the long-insurer/short-market trade loses (**-1.89 %/yr** net), and the apparent engine is sector beta you can rent more cheaply — and at a shallower drawdown — by just owning the market. Buffett's float was real; a plain insurer basket does not inherit it. *(One true aside: insurers did beat **banks** by +2.89 %/yr — but only at t = 0.96, and that's a different claim.)*